# Examples of running HypoXPy workflow to locate earthquakes
The original event catalog and phase picks are the output of the QuakeFlow with GAMMA associator.

In [1]:
import os
import numpy as np
from hypoxpy import utils
from obspy import UTCDateTime

/usr/local/anaconda3/envs/hypox/lib/python3.11/site-packages/obspy/core/util/base.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. Setup key driving parameters

In [2]:
#================= User-defined parameters =================#
binpath='/Users/xtyang/bin' #path to hypoDD binaries.
indir='input'
outdir='output'
if not os.path.exists(outdir):
    os.makedirs(outdir)
namebase='GAMMA'

"""
Data files
"""
station_file=os.path.join(indir,'GAMMA_station_list.json')
station_file_hypoinv=os.path.join(indir,'GAMMA_station_hypoinv.dat')
station_file_hypodd=os.path.join(indir,'GAMMA_station_hypodd.dat')

event_file=os.path.join(indir,'GAMMA_catalog.csv')
phase_file=os.path.join(indir,'GAMMA_picks.csv') #input original phase file.
phase_file_reformat_hypoinv=f'{indir}/{namebase}_phase_hypoinv.pha' #output reformatted phase file for hypoDD.
phase_file_reformat_hypodd=f'{indir}/{namebase}_phase_hypodd.pha' #output reformatted phase file for hypoDD.

#final catalogs
#file names for the final summary catalogs.
out_hypoinv_bad = '%s/%s_hypoinv_bad.csv'%(outdir,namebase)
out_hypoinv_good = '%s/%s_hypoinv_good.csv'%(outdir,namebase)
out_hypodd_final = '%s/%s_hypodd_catalog.csv'%(outdir,namebase)

evid_label='event_id' #column name for event ID in the input event and pick files.
mapping_evid=True #whether to remap event IDs to integers starting from 1 for HypoDD compatibility.
        #If False, original event IDs are used (may cause issues if IDs are not integers or not starting from 1).
        #THIS SHOULD BE TRUE, UNLESS YOU ARE SURE YOUR EVENT IDS ARE INTEGERS STARTING FROM 1.
if mapping_evid:
      evid_label_mapped=evid_label + "_mapped" #this will be used only if mapping_evid is True.
else:
      evid_label_mapped = evid_label

save_cleaned_data=True #whether to save cleaned event and pick data after removing invalid entries.
        #THIS SHOULD BE TRUE IF mapping_evid IS TRUE. IT IS SET AUTOMATICALLY IN THE FUNCTION IF mapping_evid IS TRUE.
cleaned_eventfile = os.path.splitext(event_file)[0] + "_cleaned.csv"
cleaned_pickfile = os.path.splitext(phase_file)[0] + "_cleaned.csv"

# combine network and station for HypoDD format.
combine_net_sta=True
cleanup=True #remove intermediate files from each relocation run if True.
qc_phase=False #QC phase when reformatting. Currently, only check if both P and S are available. Recommend to set to False.

"""
Control parameters for running HypoInverse
"""
pmodel=os.path.join(indir,'velo_p_eg.cre')
smodel=os.path.join(indir,'velo_s_eg.cre')

depth_try_list=np.arange(0,20.5,1.0) #list of depths to try for each event during grid search.

min_nsta=4 #minimum number of stations to relocate the earthquake. 4 is recommeneded as the minimum to get a reliable location.
template_par_hypoinv = os.path.join(indir,'template_hypoinv_par.inp')

"""
Control parameters for running HypoDD
"""
#depth correction for phase file reformatting to avoid air quakes.
dep_corr = 5  #km, modify velocity model accordingly.

#change the template parameters for more controls on hypoinverse, hypodd, and ph2dt.

template_par_ph2dt = os.path.join(indir,'template_ph2dt_par.inp')
template_par_hypodd = os.path.join(indir,'template_hypodd_par.inp')


## 2. Call the main driver

In [ ]:
# test converting from hypoinverse phase file to hypodd phase file
utils.convert_hypoinverse2hypodd(phase_file_reformat_hypoinv,phase_file_reformat_hypodd+".test",verbose=True) 

[WARNING] convert_hypoinverse2hypodd() is untested. Use with caution.
[INFO] Wrote single phase file: input/GAMMA_phase_hypodd.pha.test
[INFO] Number of events: 9257
